# Day 9 — Random Forest Hyperparameter Tuning and Model Complexity Analysis

## Objective
Investigate how Random Forest hyperparameters affect telecom network anomaly detection performance, generalization capability, and tree complexity using the public **TelecomTS** dataset.

## Research Questions
1. **Can shallower trees maintain strong anomaly-detection performance?**  
   Does constraining tree depth (`max_depth=10` or `20`) mitigate model complexity without sacrificing anomaly detection sensitivity?
2. **How do minimum leaf size and feature subsampling affect the model?**  
   Does increasing `min_samples_leaf` regularize the model effectively, or does it prune away rare, critical fault signatures under 3.86% class imbalance? Does increasing `max_features` from `sqrt` (8 KPIs) to `0.5` (32 KPIs) provide better splits or simply inflate computational cost?
3. **Can model complexity be reduced without substantially sacrificing anomaly recall and F1-score?**  
   What is the trade-off between total tree nodes, training latency, and fault coverage?

## Experimental Methodology
- **Dataset**: Public TelecomTS cellular network KPI dataset (`AliMaatouk/TelecomTS` on Hugging Face).
- **Feature Engineering**: 64 statistical features across 16 numerical radio and traffic KPIs (Mean, Standard Deviation with $ddof=1$, Min, Max).
- **Train/Test Partitioning**: Stratified 80/20 train-test split (`random_state=42`, $N_{\text{train}}=25,600$, $N_{\text{test}}=6,400$).
- **Hyperparameter Tuning**: 5-Fold Stratified Cross-Validation strictly on the training set ($N=25,600$), using anomaly-class F1-score as the selection criterion.
- **Untouched Evaluation**: Final selected model evaluated once on the reserved test set ($N=6,400$) and compared head-to-head with the Day 8 baseline.

## Academic & Scope Disclaimer
This prototype is an independent academic research project utilizing open-access cellular KPI data from TelecomTS. The results do not reflect proprietary operational data, infrastructure architecture, or measured network faults on SLTMobitel's commercial cellular network.

In [1]:
# Setup cell for Google Colab environments
# Installs Hugging Face datasets library silently if not already present
!pip install -q datasets

## Step 2 — Load TelecomTS Dataset

We load the public **TelecomTS** dataset directly from Hugging Face (`AliMaatouk/TelecomTS`).
Loading directly from the hub ensures the notebook runs independently from start to finish in Google Colab without requiring external file transfers.

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "AliMaatouk/TelecomTS",
    data_files={"full": "**/chunked.jsonl"}
)

data = dataset["full"]
print("Number of samples:", len(data))
print("Dataset columns:", data.column_names)

Number of samples: 32000
Dataset columns: ['start_time', 'end_time', 'sampling_rate', 'KPIs', 'description', 'anomalies', 'statistics', 'labels', 'QnA']


## Step 3 — Feature Extraction & KPI Definitions

We summarize each of the **16 numerical KPIs** using **4 statistical metrics** across its observation window:
1. **Mean**: Average signal or throughput level.
2. **Standard Deviation (`std`)**: Signal stability and dispersion ($ddof=1$, sample standard deviation).
3. **Minimum (`min`)**: Lowest observed trough.
4. **Maximum (`max`)**: Peak spike value.

$$\text{Total Features} = 16 \text{ KPIs} \times 4 \text{ metrics} = 64 \text{ features}$$

Target encoding: **0 = Normal**, **1 = Anomaly**.

In [3]:
import pandas as pd
import numpy as np

numerical_kpis = [
    "RSRP", "DL_BLER", "DL_MCS", "UL_BLER", "UL_MCS",
    "UL_NPRB", "UL_SNR", "TX_Bytes", "RX_Bytes", "Estimated_UL_Buffer",
    "PRBs_DL_Current", "PRBs_UL_Current", "PRB_Utilization_DL",
    "PRB_Utilization_UL", "UL_NumberOfPackets", "DL_NumberOfPackets"
]

feature_rows = []
for sample in data:
    row = {}
    for kpi in numerical_kpis:
        values = sample["KPIs"][kpi]
        row[f"{kpi}_mean"] = sum(values) / len(values)
        row[f"{kpi}_std"] = pd.Series(values).std()  # ddof=1, exact Day 8 logic
        row[f"{kpi}_min"] = min(values)
        row[f"{kpi}_max"] = max(values)
    feature_rows.append(row)

X = pd.DataFrame(feature_rows)
y = pd.Series([
    1 if sample["labels"]["anomaly_present"] == "Yes" else 0
    for sample in data
])

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts().sort_index())
print("\nTarget percentage:")
print((y.value_counts(normalize=True) * 100).sort_index())

Feature shape: (32000, 64)
Target shape: (32000,)

Target distribution:
0    30765
1     1235
Name: count, dtype: int64

Target percentage:
0    96.140625
1     3.859375
Name: proportion, dtype: float64


## Step 4 — Verification of Experimental Setup & Train-Test Split

Before initiating hyperparameter tuning, we verify that our experimental setup strictly aligns with the Day 8 baseline:
- **Split**: Stratified 80/20 train-test split (`test_size=0.20`, `random_state=42`, `stratify=y`).
- **Sample Counts**: 25,600 training samples (988 anomalies), 6,400 testing samples (247 anomalies).
- **Deterministic Alignment Check**: We train the reference Day 8 baseline model (`n_estimators=100`, `random_state=42`, `n_jobs=-1`) and verify that it reproduces the exact recorded test set confusion matrix ($TN=6153, FP=0, FN=17, TP=230$, $F1=96.44\%$, $ROC-AUC=0.9998$, $PR-AUC=0.9954$).

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, average_precision_score

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)} (Anomaly rate: {y_train.mean():.4f})")
print(f"Testing samples: {len(X_test)} (Anomaly rate: {y_test.mean():.4f})")

# Ground truth reproduction check
baseline_check = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
baseline_check.fit(X_train, y_train)
base_pred = baseline_check.predict(X_test)
base_prob = baseline_check.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, base_pred)
tn, fp, fn, tp = cm.ravel()
print("\n--- Baseline Reproduction Check on Test Set ---")
print("Confusion Matrix:\n", cm)
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")
print(f"ROC-AUC: {roc_auc_score(y_test, base_prob):.4f}")
print(f"PR-AUC: {average_precision_score(y_test, base_prob):.4f}")

assert tn == 6153 and fp == 0 and fn == 17 and tp == 230, "Split mismatch!"
print("\n>> VERIFICATION PASSED: Experimental setup and test samples match Day 8 exactly. <<")

Training samples: 25600 (Anomaly rate: 0.0386)
Testing samples: 6400 (Anomaly rate: 0.0386)

--- Baseline Reproduction Check on Test Set ---
Confusion Matrix:
[[6153    0]
 [  17  230]]
True Negatives (TN): 6153
False Positives (FP): 0
False Negatives (FN): 17
True Positives (TP): 230
ROC-AUC: 0.9998
PR-AUC: 0.9954

>> VERIFICATION PASSED: Experimental setup and test samples match Day 8 exactly. <<


## Step 5 — Hyperparameter Tuning Strategy

### Why Cross-Validation on the Training Set Only?
- **Leakage Prevention**: Tuning hyperparameters directly against the test set leads to optimistic bias ("data snooping"). We use **5-fold Stratified Cross-Validation strictly on the 25,600 training samples**.
- **Class Preservation**: `StratifiedKFold` ensures each fold preserves the true 3.86% anomaly rate (~198 anomalies per validation fold).
- **Selection Criterion**: Anomaly-class F1-score (`scoring='f1'`). In telecom anomaly monitoring, balanced harmonic precision-recall is far more informative than overall accuracy under heavy imbalance.

### Hyperparameter Grid (18 Combinations x 5 Folds = 90 Total Fits):
- `max_depth`: `[10, 20, None]` (Evaluates shallow, moderate, and fully grown trees)
- `min_samples_leaf`: `[1, 2, 5]` (Evaluates regularization via leaf sample minimums)
- `max_features`: `["sqrt", 0.5]` (Evaluates standard 8-KPI random subspace vs. 32-KPI broad splits)
- Fixed: `n_estimators=100`, `random_state=42`, `n_jobs=-1`.

In [5]:
import time
from sklearn.model_selection import StratifiedKFold, GridSearchCV

param_grid = {
    "max_depth": [10, 20, None],
    "min_samples_leaf": [1, 2, 5],
    "max_features": ["sqrt", 0.5]
}

base_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc"
}

print("Executing GridSearchCV (5-Fold Stratified CV, 90 fits)...")
cv_start = time.time()
grid_search = GridSearchCV(
    estimator=base_rf,
    param_grid=param_grid,
    cv=cv,
    scoring=scoring,
    refit="f1",
    n_jobs=1,
    verbose=1
)
grid_search.fit(X_train, y_train)
cv_duration = time.time() - cv_start

print(f"Grid search completed in {cv_duration:.2f} seconds ({cv_duration/60:.2f} minutes).")
print(f"\nBest Hyperparameters (selected by CV F1): {grid_search.best_params_}")
print(f"Best Cross-Validation F1-score: {grid_search.best_score_:.4f} ({grid_search.best_score_*100:.2f}%)")

Executing GridSearchCV (5-Fold Stratified CV, 90 fits)...
Grid search completed in 1371.95 seconds (22.87 minutes).

Best Hyperparameters (selected by CV F1): {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1}
Best Cross-Validation F1-score: 0.9601 (96.01%)


## Step 6 — Cross-Validation Results Analysis

Let us inspect the full rankings of all 18 hyperparameter combinations based on their out-of-fold validation performance across the 5 cross-validation splits.

In [6]:
cv_results = pd.DataFrame(grid_search.cv_results_)

summary_cols = [
    "param_max_depth", "param_min_samples_leaf", "param_max_features",
    "mean_test_f1", "std_test_f1",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_roc_auc", "mean_fit_time"
]
cv_summary = cv_results[summary_cols].copy()
cv_summary["param_max_depth"] = cv_summary["param_max_depth"].fillna("None")
cv_summary = cv_summary.sort_values(by="mean_test_f1", ascending=False).reset_index(drop=True)
cv_summary.index = cv_summary.index + 1

print(cv_summary.to_string())

Rank  max_depth  min_samples_leaf max_features   Mean F1    Std F1  Mean Precision  Mean Recall  Mean ROC-AUC  Mean Fit Time
   1       None                 1         sqrt  0.960059  0.005537        0.997838     0.925109      0.998285        8.06 s
   2       None                 2         sqrt  0.955706  0.006779        0.996745     0.918023      0.997301       10.26 s
   3       None                 1          0.5  0.954188  0.004861        0.994583     0.917013      0.996082       31.01 s
   4       None                 2          0.5  0.954182  0.005713        0.994583     0.917018      0.996150       33.03 s
   5         20                 1         sqrt  0.947297  0.008414        0.997796     0.901825      0.997578        5.78 s
   6         20                 1          0.5  0.945750  0.006642        0.995616     0.900805      0.992186       25.63 s
   7       None                 5          0.5  0.945217  0.007407        0.995568     0.899805      0.994824       29.66 s
   8   

In [7]:
import matplotlib.pyplot as plt

# Visualization 1: Hyperparameter Ranking by CV F1-score
plt.figure(figsize=(11, 7))
labels = [
    f"d={row['param_max_depth']}, l={row['param_min_samples_leaf']}, f={row['param_max_features']}"
    for _, row in cv_summary.iterrows()
]
colors = ["#2ca02c" if i == 0 else "#1f77b4" for i in range(len(cv_summary))]
plt.barh(labels[::-1], cv_summary["mean_test_f1"][::-1], color=colors[::-1], xerr=cv_summary["std_test_f1"][::-1], capsize=3)
plt.xlabel("Mean 5-Fold Cross-Validation Anomaly F1-Score")
plt.ylabel("Hyperparameter Candidates (depth, leaf, features)")
plt.title("Random Forest Hyperparameter Tuning: 5-Fold Stratified CV F1-Score")
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## Step 7 — Final Evaluation of Selected Model on Untouched Test Set

The hyperparameter search through 5-fold stratified cross-validation on the training set identified:
- `max_depth`: `None`
- `max_features`: `'sqrt'`
- `min_samples_leaf`: `1`

> **Key Scientific Finding:**  
> Cross-validation independently selected the exact Day 8 baseline configuration as the empirical optimum. We now evaluate this model once on the unseen test set ($N=6,400$).

In [8]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score

best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)
y_prob_tuned = best_model.predict_proba(X_test)[:, 1]

print("Tuned Model Test Classification Report:")
print(classification_report(y_test, y_pred_tuned, target_names=["Normal", "Anomaly"], digits=4))

cm_tuned = confusion_matrix(y_test, y_pred_tuned)
print("Confusion Matrix:\n", cm_tuned)
tn, fp, fn, tp = cm_tuned.ravel()
print(f"True Negatives (TN): {tn}")
print(f"False Positives (FP): {fp}")
print(f"False Negatives (FN): {fn}")
print(f"True Positives (TP): {tp}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_tuned):.4f}")
print(f"PR-AUC (Average Precision): {average_precision_score(y_test, y_prob_tuned):.4f}")

Tuned Model Test Classification Report:
              precision    recall  f1-score   support

      Normal     0.9972    1.0000    0.9986      6153
     Anomaly     1.0000    0.9312    0.9644       247

    accuracy                         0.9973      6400
   macro avg     0.9986    0.9656    0.9815      6400
weighted avg     0.9974    0.9973    0.9973      6400

Confusion Matrix:
[[6153    0]
 [  17  230]]
True Negatives (TN): 6153
False Positives (FP): 0
False Negatives (FN): 17
True Positives (TP): 230
ROC-AUC: 0.9998
PR-AUC (Average Precision): 0.9954


## Step 8 — Model Complexity & Head-to-Head Comparison

We measure tree complexity using:
1. **Maximum Observed Tree Depth**: The longest root-to-leaf decision path across all 100 trees.
2. **Mean Tree Depth**: The average depth of the 100 trees.
3. **Total Tree Node Count**: Cumulative split and leaf nodes across all 100 trees.

In [9]:
tuned_depths = [e.get_depth() for e in best_model.estimators_]
tuned_nodes = [e.tree_.node_count for e in best_model.estimators_]

comparison_df = pd.DataFrame({
    "Metric / Property": [
        "Hyperparameters",
        "Cross-Validation F1-score",
        "Test Precision (Anomaly)",
        "Test Recall (Anomaly)",
        "Test F1-score (Anomaly)",
        "Test ROC-AUC",
        "Test PR-AUC",
        "True Positives (TP)",
        "False Positives (FP)",
        "False Negatives (FN)",
        "True Negatives (TN)",
        "Max Tree Depth",
        "Mean Tree Depth",
        "Total Tree Nodes",
        "Mean Nodes per Tree",
        "Full Fit Duration",
        "CV Search Duration (90 fits)"
    ],
    "Day 8 Baseline RF": [
        "max_depth=None, min_samples_leaf=1, max_features='sqrt'",
        "N/A (Evaluated directly on test)",
        "100.00%", "93.12%", "96.44%", "0.9998", "0.9954",
        "230", "0", "17", "6153",
        "44", "30.80", "29,136", "291.36", "11.26 s", "N/A"
    ],
    "Day 9 Tuned RF": [
        str(grid_search.best_params_),
        f"{grid_search.best_score_*100:.2f}%",
        "100.00%", "93.12%", "96.44%", "0.9998", "0.9954",
        "230", "0", "17", "6153",
        f"{max(tuned_depths)}", f"{np.mean(tuned_depths):.2f}",
        f"{sum(tuned_nodes):,}", f"{np.mean(tuned_nodes):.2f}",
        f"{grid_search.refit_time_:.2f} s",
        f"{cv_duration:.2f} s ({cv_duration/60:.2f} min)"
    ]
})

print("=" * 88)
print("HEAD-TO-HEAD COMPARISON: DAY 8 BASELINE VS. DAY 9 TUNED RANDOM FOREST")
print("=" * 88)
print(comparison_df.to_string(index=False))

HEAD-TO-HEAD COMPARISON: DAY 8 BASELINE VS. DAY 9 TUNED RANDOM FOREST
Metric / Property             Day 8 Baseline RF                                                     Day 9 Tuned RF                                                        
Hyperparameters               max_depth=None, min_samples_leaf=1, max_features='sqrt'               {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1}  
Cross-Validation F1-score     N/A (Evaluated directly on test)                                      96.01%                                                                
Test Precision (Anomaly)      100.00%                                                               100.00%                                                               
Test Recall (Anomaly)         93.12%                                                                93.12%                                                                
Test F1-score (Anomaly)       96.44%                                         

## Day 9 Conclusions & Defense Insights

### Answers to Research Questions:

1. **Can shallower trees maintain strong anomaly-detection performance?**  
   **No.** Restricting `max_depth=10` severely degraded anomaly recall (dropping from 92.51% down to 80.67%–85.53%) and F1-score (dropping from 96.01% to 89.23%–92.08%). Moderately deep trees (`max_depth=20`) retained better performance (F1 ~94.73%), but still missed ~2.3% more real anomalies. Because anomalies represent only 3.86% of traffic, decision paths require sufficient depth (~30–44 splits) to isolate rare multi-KPI interaction boundaries.

2. **How do minimum leaf size and feature subsampling affect the model?**  
   - **Leaf Size (`min_samples_leaf`)**: Increasing leaf size from 1 to 2 or 5 acts as strong regularization. While it preserves near-zero false alarms (precision >99.5%), it prunes away small clusters of anomalous patterns, decreasing recall.
   - **Feature Subsampling (`max_features`)**: Evaluating 50% of features (32 KPIs) per split instead of `'sqrt'` (8 KPIs) provided no performance gain (CV F1: 95.42% vs 96.01%) while inflating training time by nearly 400% (from 8.06 s to 31.01 s per fold). Subspace decorrelation with `sqrt` is both faster and more accurate.

3. **Can model complexity be reduced without substantially sacrificing anomaly recall and F1-score?**  
   **No.** In this tabular KPI representation, full tree depth without leaf pruning is necessary to preserve the peak 93.12% anomaly recall. The unconstrained ensemble of 100 trees with `max_features='sqrt'` and `min_samples_leaf=1` remains the empirical global optimum.

### Viva Defense Note: Honest Model Evaluation
In applied machine learning, hyperparameter tuning does not always yield higher metrics over an already strong baseline. Demonstrating through rigorous 5-fold cross-validation that the baseline parameters were already optimal proves methodological maturity and guards against the common trap of claiming illusory improvements.